In [5]:
import os
import time
import requests
import pandas as pd

# ================== CONFIG ==================

SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ASTRUSDT"]
INTERVAL = "1h"

START_DATE = "2020-01-01"
END_DATE   = "2025-01-01"

BASE_URL = "https://api.binance.com"

# Main folder for all crypto data
MAIN_DIR = "crypto_data"  # Auto-created


# ================== HELPERS ==================

def to_ms(date_str):
    """Convert YYYY-MM-DD to milliseconds timestamp."""
    return int(pd.Timestamp(date_str).timestamp() * 1000)


def get_binance_ohlcv(symbol, interval, start_time, end_time):
    """Download OHLCV data from Binance using pagination."""
    url = f"{BASE_URL}/api/v3/klines"
    all_data = []
    start = start_time

    while True:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": start,
            "endTime": end_time,
            "limit": 1000
        }
        data = requests.get(url, params=params).json()
        if not data:
            break

        all_data.extend(data)
        start = data[-1][0] + 1
        time.sleep(0.2)

    cols = [
        "open_time","open","high","low","close","volume","close_time",
        "quote_asset_volume","trades","taker_buy_volume",
        "taker_buy_quote_volume","ignore"
    ]

    df = pd.DataFrame(all_data, columns=cols)
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

    float_cols = ["open","high","low","close","volume",
                  "quote_asset_volume","taker_buy_volume","taker_buy_quote_volume"]

    for col in float_cols:
        df[col] = df[col].astype(float)

    df["trades"] = df["trades"].astype(int)

    return df


def add_technical_indicators(df):
    """Add TA indicators."""
    close = df["close"]
    volume = df["volume"]

    df["SMA_20"] = close.rolling(20).mean()
    df["EMA_20"] = close.ewm(span=20, adjust=False).mean()

    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / avg_loss

    df["RSI_14"] = 100 - (100 / (1 + rs))

    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

    df["BBM"] = close.rolling(20).mean()
    std_20 = close.rolling(20).std()
    df["BBU"] = df["BBM"] + 2 * std_20
    df["BBL"] = df["BBM"] - 2 * std_20

    obv = [0]
    for i in range(1, len(df)):
        if close.iloc[i] > close.iloc[i-1]:
            obv.append(obv[-1] + volume.iloc[i])
        elif close.iloc[i] < close.iloc[i-1]:
            obv.append(obv[-1] - volume.iloc[i])
        else:
            obv.append(obv[-1])

    df["OBV"] = obv
    return df


def add_ml_features(df):
    """Add lag features + returns + volatility + targets."""
    for lag in [1,3,6,12,24]:
        df[f"close_lag_{lag}"] = df["close"].shift(lag)
        df[f"volume_lag_{lag}"] = df["volume"].shift(lag)

    df["return_1h"] = df["close"].pct_change(1)
    df["return_3h"] = df["close"].pct_change(3)
    df["return_6h"] = df["close"].pct_change(6)

    for w in [3,6,12,24]:
        df[f"vol_{w}h"] = df["close"].rolling(w).std()

    df["target_next_close"] = df["close"].shift(-1)
    df["target_up_down"] = (df["target_next_close"] > df["close"]).astype(int)

    return df


def process_symbol(symbol):
    """Download → process → save into unique folder."""

    print(f"\n==============================")
    print(f"Processing {symbol} ...")
    print(f"==============================")

    # Create main folder
    os.makedirs(MAIN_DIR, exist_ok=True)

    # Create subfolder for this symbol
    symbol_dir = os.path.join(MAIN_DIR, symbol)
    os.makedirs(symbol_dir, exist_ok=True)

    start_ms = to_ms(START_DATE)
    end_ms = to_ms(END_DATE)

    df = get_binance_ohlcv(symbol, INTERVAL, start_ms, end_ms)
    df = add_technical_indicators(df)
    df = add_ml_features(df)

    df = df.dropna().reset_index(drop=True)

    file_path = os.path.join(symbol_dir, f"{symbol}_ML_ready.csv")
    df.to_csv(file_path, index=False)

    print(f"Saved: {file_path}")
    print(f"Shape: {df.shape}")


# ================== MAIN ==================

if __name__ == "__main__":
    for sym in SYMBOLS:
        process_symbol(sym)
        time.sleep(0.5)



Processing BTCUSDT ...
Saved: crypto_data\BTCUSDT\BTCUSDT_ML_ready.csv
Shape: (43792, 40)

Processing ETHUSDT ...
Saved: crypto_data\ETHUSDT\ETHUSDT_ML_ready.csv
Shape: (43792, 40)

Processing BNBUSDT ...
Saved: crypto_data\BNBUSDT\BNBUSDT_ML_ready.csv
Shape: (43792, 40)

Processing XRPUSDT ...
Saved: crypto_data\XRPUSDT\XRPUSDT_ML_ready.csv
Shape: (43792, 40)

Processing ASTRUSDT ...
Saved: crypto_data\ASTRUSDT\ASTRUSDT_ML_ready.csv
Shape: (24876, 40)
